# V6 on Google Colab (T4 GPU)

This notebook mirrors the Kaggle template but sources data from Google
Drive, because Colab does not natively host datasets.

## 1. Mount Drive and locate the ABT

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/bpm-v6  # expects abt_v6_cached.parquet + src_and_scripts.zip

## 2. Install dependencies & unpack source

In [ ]:
!pip install -q lightgbm==4.6.0 joblib pyarrow matplotlib scikit-learn optuna
import zipfile, os, sys, shutil
os.makedirs('/content/repo/output', exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/bpm-v6/src_and_scripts.zip') as z:
    z.extractall('/content/repo')
shutil.copy('/content/drive/MyDrive/bpm-v6/abt_v6_cached.parquet', '/content/repo/output/')
sys.path.insert(0, '/content/repo')
os.chdir('/content/repo')
print('ready:', os.listdir('/content/repo')[:10])

## 3. Train V6

In [ ]:
!nvidia-smi | head -3
!python -m scripts.train_v6 --num-boost-round 1500

## 4. Rolling-origin CV

In [ ]:
!python -m scripts.rolling_origin_cv --abt output/abt_v6_cached.parquet \
    --target target_qty_imputed --reg-objective pinball --alpha 0.6 \
    --n-origins 8 --num-boost-round 800 \
    --output-json output/v6_rolling_cv_gpu.json --output-md output/v6_rolling_cv_gpu.md

## 5. Save artefacts back to Drive

In [ ]:
dst = '/content/drive/MyDrive/bpm-v6/artefacts/'
os.makedirs(dst, exist_ok=True)
for f in ['output/model_v6.joblib', 'output/v6_rolling_cv_gpu.json',
          'output/v6_rolling_cv_gpu.md', 'output/feature_importance_v6.csv',
          'output/v6_metrics.csv', 'output/v6_vs_v5.md']:
    if os.path.exists(f):
        shutil.copy(f, dst)
print('done:', os.listdir(dst))